# Lakebase Setup for MMF Forecast Serving

This notebook provisions a **Lakebase Provisioned** (managed PostgreSQL) instance and creates
**synced tables** that mirror the forecast output Delta tables. The Dash app in `apps/` can then
read from PostgreSQL for low-latency OLTP access instead of hitting the SQL warehouse.

**Steps:**
1. Create the Lakebase instance
2. Enable Change Data Feed on source tables
3. Create synced tables (reverse ETL)
4. Verify connectivity

Run this notebook once on any compute (including serverless).

In [0]:
INSTANCE_NAME = "mmf-forecast-serving"   # Lakebase instance name
CAPACITY = "CU_1"                         # CU_1, CU_2, CU_4, CU_8

CATALOG = "mmf_demo_aym"
SCHEMA = "m4_advanced"

# Tables to sync from Delta to Lakebase
SYNC_TABLES = [
    "monthly_evaluation_output",
    "monthly_scoring_output",
    "m4_monthly_train",
]

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Check if instance already exists
existing = {i.name for i in w.database.list_database_instances()}
if INSTANCE_NAME in existing:
    inst = w.database.get_database_instance(name=INSTANCE_NAME)
    print(f"Instance '{INSTANCE_NAME}' already exists — state: {inst.state}")
else:
    inst = w.database.create_database_instance(
        name=INSTANCE_NAME, capacity=CAPACITY, stopped=False,
    )
    print(f"Created instance '{INSTANCE_NAME}'")

print(f"DNS: {inst.read_write_dns}")
print(f"State: {inst.state}")

In [0]:
for table in SYNC_TABLES:
    fqn = f"`{CATALOG}`.`{SCHEMA}`.`{table}`"
    spark.sql(f"ALTER TABLE {fqn} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    print(f"CDF enabled on {fqn}")

In [0]:
# Register the Lakebase instance under the same catalog so synced tables
# appear as catalog.schema.table in UC.
try:
    w.database.register_database_instance(
        name=INSTANCE_NAME, catalog=CATALOG, schema=SCHEMA,
    )
    print(f"Registered {INSTANCE_NAME} under {CATALOG}.{SCHEMA}")
except Exception as e:
    if "ALREADY_EXISTS" in str(e) or "already registered" in str(e).lower():
        print(f"Instance already registered under UC — skipping")
    else:
        raise

In [0]:
from databricks.sdk.service.database import (
    SyncedDatabaseTable,
    SyncedTableSpec,
    SyncedTableSchedulingPolicy,
)

for table in SYNC_TABLES:
    source_fqn = f"{CATALOG}.{SCHEMA}.{table}"
    target_fqn = f"{CATALOG}.{SCHEMA}.lb_{table}"  # prefix with lb_ to distinguish
    try:
        synced = w.database.create_synced_database_table(
            SyncedDatabaseTable(
                name=target_fqn,
                database_instance_name=INSTANCE_NAME,
                spec=SyncedTableSpec(
                    source_table_full_name=source_fqn,
                    scheduling_policy=SyncedTableSchedulingPolicy.TRIGGERED,
                ),
            )
        )
        print(f"Synced table created: {source_fqn} -> {target_fqn}")
    except Exception as e:
        if "ALREADY_EXISTS" in str(e):
            print(f"Synced table {target_fqn} already exists — skipping")
        else:
            print(f"Error syncing {source_fqn}: {e}")

In [0]:
import psycopg
import uuid

cred = w.database.generate_database_credential(
    request_id=str(uuid.uuid4()),
    instance_names=[INSTANCE_NAME],
)
username = w.current_user.me().user_name

conn_str = (
    f"host={inst.read_write_dns} dbname=postgres "
    f"user={username} password={cred.token} sslmode=require"
)

with psycopg.connect(conn_str) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version()")
        print(f"Connected: {cur.fetchone()[0]}")
        cur.execute("SELECT tablename FROM pg_tables WHERE schemaname = 'public'")
        tables = [row[0] for row in cur.fetchall()]
        print(f"Tables in Lakebase: {tables}")

## Next Steps

1. **Deploy the app** with the Lakebase resource:
   ```bash
   databricks apps create mmf-forecast-app --source-code-path apps/
   databricks apps add-resource mmf-forecast-app \
       --resource-type database \
       --resource-name lakebase \
       --database-instance mmf-forecast-serving
   ```
2. **Trigger a sync** after each forecast run to refresh Lakebase data
3. **Monitor sync status** in Catalog Explorer under the synced tables